<a href="https://colab.research.google.com/github/kdh2002/kfa-ai-agent/blob/main/w02_%EB%82%B4%EC%86%8C%EC%8A%A4%EC%88%98%EC%A7%91.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import io
import zipfile
import xml.etree.ElementTree as ET
from getpass import getpass

import pandas as pd
import requests
from IPython.display import display

API_KEY = getpass("OpenDART API 키(40자리)를 입력하세요: ").strip()
if len(API_KEY) != 40:
    raise ValueError("OpenDART API 키는 일반적으로 40자리입니다. 발급받은 키를 확인하세요.")
print("API 키 입력 완료 (키 값은 출력하지 않음)")

OpenDART API 키(40자리)를 입력하세요: ··········
API 키 입력 완료 (키 값은 출력하지 않음)


In [5]:
url = "https://opendart.fss.or.kr/api/corpCode.xml"
response = requests.get(url, params={"crtfc_key": API_KEY}, timeout=60)
response.raise_for_status()

buffer = io.BytesIO(response.content)
if not zipfile.is_zipfile(buffer):
    # 인증 오류라면 보통 ZIP 대신 오류 메시지가 돌아온다.
    raise RuntimeError("기업코드 다운로드 실패: " + response.text[:250])

with zipfile.ZipFile(buffer) as archive:
    xml_filename = next(name for name in archive.namelist() if name.lower().endswith('.xml'))
    root = ET.fromstring(archive.read(xml_filename))

matches = [item for item in root.findall('list') if item.findtext('stock_code', '').strip() == '016360']
if len(matches) != 1:
    raise ValueError(f"종목코드 016360 검색 결과가 {len(matches)}개입니다. 확인이 필요합니다.")

corp_code = matches[0].findtext('corp_code').strip()
corp_name = matches[0].findtext('corp_name').strip()
print('기업명:', corp_name)
print('종목코드: 016360')
print('DART 고유번호:', corp_code)

기업명: 삼성증권
종목코드: 016360
DART 고유번호: 00104856


In [6]:
BASE_URL = "https://opendart.fss.or.kr/api/list.json"
START_DATE = "20250101"
END_DATE = "20261101"

records = []
page = 1

while True:
    params = {
        "crtfc_key": API_KEY,
        "corp_code": corp_code,
        "bgn_de": START_DATE,
        "end_de": END_DATE,
        "pblntf_ty": "I",        # 거래소공시
        "pblntf_detail_ty": "I002",  # 공정공시
        "page_no": page,
        "page_count": 100,
        "sort": "date",
        "sort_mth": "desc",
    }
    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    status = data.get("status")

    if status == "013":
        print("조회된 공정공시가 없습니다. 검색기간이나 공시 유형을 확인하세요.")
        break
    if status != "000":
        raise RuntimeError(f"OpenDART 오류: {status} / {data.get('message')}")

    records.extend(data.get("list", []))
    if page >= int(data.get("total_page", 1)):
        break
    page += 1

print(f"수집한 공정공시 전체: {len(records)}건")

수집한 공정공시 전체: 51건


In [7]:
columns = ["회사명", "접수일", "보고서명", "접수번호", "원본링크"]
rows = [
    {
        "회사명": item.get("corp_name", ""),
        "접수일": item.get("rcept_dt", ""),
        "보고서명": item.get("report_nm", ""),
        "접수번호": item.get("rcept_no", ""),
        "원본링크": f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={item.get('rcept_no', '')}",
    }
    for item in records
]
all_df = pd.DataFrame(rows, columns=columns)

# '영업(잠정)실적' 제목이 가장 중요하다. '영업실적'도 누락 점검용 후보로 표시한다.
mask = all_df["보고서명"].str.contains(r"영업\(잠정\)실적|영업.*실적", regex=True, na=False)
result_df = all_df[mask].copy().reset_index(drop=True)

print(f"전체 공정공시: {len(all_df)}건")
print(f"영업실적 관련 후보: {len(result_df)}건")
if result_df.empty:
    print("일치하는 제목이 없으므로 전체 공정공시 제목을 확인하세요.")
    display(all_df)
else:
    display(result_df)

전체 공정공시: 51건
영업실적 관련 후보: 5건


,회사명,접수일,보고서명,접수번호,원본링크
0,삼성증권,20260804,연결재무제표기준영업(잠정)실적(공정공시),20260804800508,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
1,삼성증권,20260511,연결재무제표기준영업(잠정)실적(공정공시),20260511800572,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
2,삼성증권,20251111,연결재무제표기준영업(잠정)실적(공정공시),20251111800414,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
3,삼성증권,20250808,연결재무제표기준영업(잠정)실적(공정공시),20250808800368,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
4,삼성증권,20250514,연결재무제표기준영업(잠정)실적(공정공시),20250514800727,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...


In [8]:
output = 'w02_삼성증권_공정공시목록.csv'
all_df.to_csv(output, index=False, encoding='utf-8-sig')
print('CSV 저장:', output)
print('조별 표에 기록할 내용:')
print('- 담당 소스: DART 영업(잠정)실적 공정공시')
print('- 수집 단계: 1 (공식 API)')
print(f'- 수집 결과: 공정공시 {len(all_df)}건, 영업실적 관련 후보 {len(result_df)}건')
print('- 추가 확인: 관련 공시 원문에서 실적 숫자 추출, 정정 여부 확인')

CSV 저장: w02_삼성증권_공정공시목록.csv
조별 표에 기록할 내용:
- 담당 소스: DART 영업(잠정)실적 공정공시
- 수집 단계: 1 (공식 API)
- 수집 결과: 공정공시 51건, 영업실적 관련 후보 5건
- 추가 확인: 관련 공시 원문에서 실적 숫자 추출, 정정 여부 확인
